## 📎 Code this notebook uses
*(how to run, and which code each step uses: [`README.md`](README.md))*

- **`code/src/lunar/`:** `constants`, `apollo_helpers`, `ephem`, `notebook`
- **`code/pipeline/`:** `fetch_diviner.py` (optional download)

# 00 — Setup and environment check

**Run this notebook first, top to bottom.** It computes no science. It checks
that your computer is ready for notebooks 01–05, which reproduce the paper:

| Notebook | Reproduces |
|---|---|
| 01_inputs | the sensors and their stability windows (SI Table S2), the model inputs (SI Table S1), Figures 1, 2 and SI Figures S1, S2, S4 |
| 02_anchor_method | how the equilibrium solver works (SI Figure S3) |
| 03_joint_fit | the main result: albedo and $K_d$ fitted together (Table 1, Figure 4, SI Table S9) |
| 04_uncertainty_and_tests | solver tests, error budget, model comparison, likelihood-ratio tests (Tables 1–3, Figure 3; SI Tables S5, S11, S14; SI Figures S5, S6) |
| 05_annual_wave_and_posterior | the annual wave, the warming after deployment, the posterior (Figures 5, 6; SI Tables S7, S8, S12) |
| 06_performance | optional: the solver in Python against C++ (SI Table S5, last row) |

Each cell prints a short report. If a cell stops with an error, its message
says what is missing and how to fix it.

## 1. Python and package versions

Python 3.10 or newer and the core libraries. A version problem is the most
common reason a notebook will not run, so it is checked first.

In [ ]:
import sys, platform
print(f"Python {sys.version.split()[0]} on {platform.system()} {platform.machine()}")
assert sys.version_info >= (3, 10), "Python 3.10+ required"

import numpy, scipy, matplotlib, pandas, numba
for mod in (numpy, scipy, matplotlib, pandas, numba):
    print(f"  {mod.__name__:<12} {mod.__version__}")

## 2. The `lunar` package and its physical constants

The physics code is the installed package `lunar` (`code/src/lunar/`). This
cell checks that the constants the study is defined by are unchanged: the
Stefan–Boltzmann constant and the Hayne et al. (2017) regolith parameters.

In [ ]:
import math
try:
    from lunar import constants as c
except ImportError:
    raise ImportError("The lunar package is not installed. In a terminal, in the repository folder, run:\n"
                      "    pip install -e \".[dev,mcmc]\"")
assert math.isclose(c.SIGMA_SB, 5.670374419e-8, rel_tol=1e-6), "SIGMA_SB constant wrong"
assert math.isclose(c.K_DEEP, 3.4e-3, rel_tol=1e-6), "Hayne K_d default is wrong"
assert math.isclose(c.K_SURFACE, 7.4e-4, rel_tol=1e-6), "Hayne K_s default is wrong"
assert math.isclose(c.H_PARAMETER, 0.06, rel_tol=1e-6), "Hayne H is wrong"
print("Constants verified:")
print(f"  sigma = {c.SIGMA_SB} W m^-2 K^-4")
print(f"  Hayne global K_d = {c.K_DEEP*1e3:.1f} mW m^-1 K^-1,  K_s = {c.K_SURFACE*1e3:.2f},  H = {c.H_PARAMETER*100:.0f} cm")

## 3. The Apollo heat-flow record

The restored 1971–1977 Apollo Heat-Flow Experiment record (Nagihara et al.,
2018) is bundled in `code/data/apollo/`. Only sensors below 80 cm are fitted
(the top is disturbed by the borestem); there must be 7 at Apollo 15 and 16 at
Apollo 17.

In [ ]:
from lunar.apollo_helpers import extract_sensor_stability
for mission, expected in [("a15", 7), ("a17", 16)]:
    obs = extract_sensor_stability(mission, min_depth_cm=80)
    n_deep = int((obs["depth_cm_all"] >= 80).sum())
    print(f"  Apollo {mission[-2:]}: {n_deep} sensors below 80 cm (expected {expected})")
    assert n_deep == expected, f"Sensor count mismatch for {mission}"

## 4. Optional downloads

Two public data sets are too large for the repository. You need them only to
**recompute** the steps that use them; the stored results and every other step
work without them.

| Data | Size | Needed for |
|---|---|---|
| Diviner surface temperatures (PDS) | ~310 MB | rerunning the Diviner comparison (SI Table S11, SI Figure S5) |
| LOLA elevation map, 16 pixels per degree (PDS) | 33 MB | SI Figure S1 (the landing-site map) |

Set `DOWNLOAD = True` and run the cell to fetch whatever is missing.

In [ ]:
DOWNLOAD = False

from lunar import notebook as nb
from lunar._bootstrap import _download
DIV = [nb.CODE / "data/diviner/gcp" / f for f in ("global_cumul_avg_cyl_20n30n_002.tab",
                                                  "global_cumul_avg_cyl_10n20n_002.tab")]
LOLA = nb.CODE / "data/lola/ldem_16.img"
LOLA_URL = ("https://pds-geosciences.wustl.edu/lro/lro-l-lola-3-rdr-v1/lrolol_1xxx/"
            "data/lola_gdr/cylindrical/img/ldem_16.img")
have = lambda files: all(f.exists() for f in files)
print(f"  Diviner tiles : {'present' if have(DIV) else 'not downloaded'}")
print(f"  LOLA map      : {'present' if LOLA.exists() else 'not downloaded'}")
if DOWNLOAD:
    if not have(DIV):
        nb.run("fetch_diviner.py")
    if not LOLA.exists():
        LOLA.parent.mkdir(parents=True, exist_ok=True)
        print("downloading", LOLA_URL)
        assert _download(LOLA_URL, LOLA, min_bytes=30_000_000), "LOLA download failed"
        print("  LOLA map downloaded")

## 5. SPICE solar geometry

The model needs the Sun's position over a lunar day. This checks that the
bundled JPL DE440 kernels load and give a unit Sun vector.

In [ ]:
import numpy as np
from lunar.ephem import et_from_iso, solar_vector_moon_frame
sv = solar_vector_moon_frame(et_from_iso(["2024-01-01T00:00:00"]))
assert abs(np.linalg.norm(sv[0]) - 1.0) < 1e-6, "should be a unit vector"
print("  SPICE kernels OK")

## 6. Stored results

The repository ships every result the paper reports, in `code/results/`. With
them, the notebooks run in "show" mode in minutes.

In [ ]:
needed = ["stability_windows.json", "joint_albedo_fit.json", "joint_fit_checks.json",
          "joint_fit_sensitivities.json", "joint_numerics_certification.json", "likelihood_ratio.json",
          "joint_chi_density.json", "annual_wave.json", "transient_warming.json", "joint_mcmc.json"]
missing = [f for f in needed if not (nb.RESULTS / f).exists()]
assert not missing, f"missing results: {missing}"
print(f"  {len(needed)} key results files present in code/results/")
print()
print("================== ALL CHECKS PASSED ==================")
print("Next: 01_inputs.ipynb")